In [0]:
%sql
select * from dev.bronze.customer limit 10 

In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def typed(name, dtype="string"):
    """Safe cast: blank -> NULL, unparseable -> NULL (never throws). Returns a column named `name`."""
    if dtype == "string":
        return F.col(name).cast("string").alias(name)
    return F.expr(f"try_cast(nullif(trim(cast(`{name}` as string)), '') as {dtype})").alias(name)

bronze_df = spark.table(f"{CATALOG}.bronze.customer")

# Metadata columns (_ingestion_timestamp, _source_file, _pipeline_run_id) are NOT selected, so they are dropped.
# _row_seq remembers the source order so "keep the first occurrence" is deterministic on every rerun.
typed_df = (bronze_df.select(
    typed("c_custkey", "int"),
    typed("c_name", "string"),
    typed("c_address", "string"),
    typed("c_nationkey", "int"),
    typed("c_phone", "string"),
    typed("c_acctbal", "decimal(18,2)"),
    typed("c_mktsegment", "string"),
    typed("c_comment", "string")
    )
    .withColumn("_row_seq", F.monotonically_increasing_id()))

bronze_count = bronze_df.count()
print(f"Bronze customer rows read: {bronze_count}")
display(typed_df.drop("_row_seq"))

In [0]:
# ── Step 2: Primary Key Validation ──
# Rule: c_custkey must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK
# First occurrence (by source order) stays in Silver; later repeats are quarantined.

pk_null = F.col("c_custkey").isNull()
pk_window = Window.partitionBy("c_custkey").orderBy("_row_seq")      # NO "desc" and NO running count: plain row_number

flagged = (typed_df
    .withColumn("is_null_pk", pk_null)
    .withColumn("_pk_rn", F.when(pk_null, F.lit(0)).otherwise(F.row_number().over(pk_window)))
    .withColumn("is_duplicate_pk", F.col("_pk_rn") > 1))

print(f"NULL_PK failures:      {flagged.filter('is_null_pk').count()}")
print(f"DUPLICATE_PK failures: {flagged.filter('is_duplicate_pk').count()}")

In [0]:
# ── Step 3: Mandatory Attribute & Domain Validation ──
# Rule: c_name and c_address are mandatory — NULL, empty, or whitespace-only values are invalid.
# Rule: c_mktsegment must be one of the 5 known TPC-H market segments.
# Failure codes: MISSING_NAME, MISSING_ADDRESS, INVALID_MKTSEGMENT

def missing(col):
    """True when the value is NULL, empty, or only whitespace."""
    return F.col(col).isNull() | (F.length(F.trim(F.col(col))) == 0)

VALID_MKTSEGMENTS = ["BUILDING", "AUTOMOBILE", "MACHINERY", "HOUSEHOLD", "FURNITURE"]

flagged = (flagged
    .withColumn("is_missing_name",    missing("c_name"))
    .withColumn("is_missing_address", missing("c_address"))
    .withColumn("is_invalid_mktsegment",
        ~F.upper(F.trim(F.col("c_mktsegment"))).isin(VALID_MKTSEGMENTS)))

print(f"MISSING_NAME failures:       {flagged.filter('is_missing_name').count()}")
print(f"MISSING_ADDRESS failures:    {flagged.filter('is_missing_address').count()}")
print(f"INVALID_MKTSEGMENT failures: {flagged.filter('is_invalid_mktsegment').count()}")

print("\nDistinct market segments found in bronze (for audit):")
display(bronze_df.select("c_mktsegment").distinct())

In [0]:
# ── Step 4: Numeric Range Validation ──
# Failure code: INVALID_ACCTBAL

flagged = flagged.withColumn(
    "is_invalid_acctbal",
    F.col("c_acctbal").isNull())

print(f"INVALID_ACCTBAL failures: {flagged.filter('is_invalid_acctbal').count()}")

In [0]:
# ── Step 5: Referential Integrity Validation ──
# customer is a CHILD of the nation table (geography hierarchy: region -> nation -> customer/supplier).
# The parent is the already-cleaned SILVER nation table (parent-before-child order).
# A NULL or unmatched key is unresolvable -> quarantined.

fk_df = flagged

nation_parent = (spark.table(f"{CATALOG}.silver.nation")
    .select(F.col("n_nationkey").alias("_nation_pk")).distinct()
    .withColumn("_nation_hit", F.lit(1)))
fk_df = (fk_df.join(nation_parent, F.col("c_nationkey") == F.col("_nation_pk"), "left")
    .withColumn("is_nation", F.col("_nation_hit").isNull())
    .drop("_nation_pk", "_nation_hit"))

flagged = fk_df

print(f"INVALID_FK_NATION failures: {flagged.filter('is_nation').count()}")

In [0]:
# ── Step 6: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──
flagged = flagged.withColumn(
    "rejection_reason",
    F.concat_ws(", ",
        F.when(F.col("is_null_pk"), F.lit("NULL_PK")),
        F.when(F.col("is_duplicate_pk"), F.lit("DUPLICATE_PK")),
        F.when(F.col("is_missing_name"), F.lit("MISSING_NAME")),
        F.when(F.col("is_missing_address"), F.lit("MISSING_ADDRESS")),
        F.when(F.col("is_invalid_mktsegment"), F.lit("INVALID_MKTSEGMENT")),
        F.when(F.col("is_invalid_acctbal"), F.lit("INVALID_ACCTBAL")),
        F.when(F.col("is_nation"), F.lit("INVALID_FK_NATION"))
    ),
)

out_cols = ["c_custkey", "c_name", "c_address", "c_nationkey", "c_phone", "c_acctbal", "c_mktsegment", "c_comment"]

silver_df     = flagged.filter(F.col("rejection_reason") == "").select(*out_cols)
quarantine_df = flagged.filter(F.col("rejection_reason") != "").select(*out_cols, "rejection_reason")

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()
print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# overwrite => rerunning gives the same result
(silver_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.customer"))
(quarantine_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.customer_quarantine"))

print("Tables written: silver.customer, silver.customer_quarantine")

In [0]:
# ── Step 7: Bronze-Silver-Quarantine Reconciliation ──
bronze_total     = spark.table(f"{CATALOG}.bronze.customer").count()
silver_total     = spark.table(f"{CATALOG}.silver.customer").count()
quarantine_total = spark.table(f"{CATALOG}.silver.customer_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != Silver ({silver_total}) + Quarantine ({quarantine_total})"
)
print("Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 8: Data Quality Report ──
# Every check is listed, even when zero rows fail.
rules = [
    ("NULL_PK", "c_custkey must not be null", "is_null_pk"),
    ("DUPLICATE_PK", "c_custkey must be unique (first occurrence kept)", "is_duplicate_pk"),
    ("MISSING_NAME", "c_name must not be null/blank", "is_missing_name"),
    ("MISSING_ADDRESS", "c_address must not be null/blank", "is_missing_address"),
    ("INVALID_MKTSEGMENT", f"c_mktsegment must be one of {', '.join(VALID_MKTSEGMENTS)}", "is_invalid_mktsegment"),
    ("INVALID_ACCTBAL", "c_acctbal must be strictly positive (> 0)", "is_invalid_acctbal"),
    ("INVALID_FK_NATION", "c_nationkey must exist in silver.nation", "is_nation")
]
dq_rows = [("customer", code, desc, flagged.filter(F.col(flag)).count(), "Quarantine") for code, desc, flag in rules]

dq_report = spark.createDataFrame(
    dq_rows, ["table", "rule_code", "check_description", "failing_rows", "action_taken"]
)
display(dq_report)

# Persist so the whole run is auditable (overwrite per table is not needed: one table per notebook)
(dq_report.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dq_report_customer"))

if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table(f"{CATALOG}.silver.customer_quarantine"))
else:
    print("\nNo quarantined records — all rows passed every check.")

print(f"\n── Silver customer ({silver_count} rows) ──")
display(spark.table(f"{CATALOG}.silver.customer").limit(20))

In [0]:
# ── Step 9: Materialized View — customers_in_debt ──
# Customers in the CLEANED silver table whose account balance is negative (c_acctbal < 0).
spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG}.silver.customers_in_debt
    COMMENT 'Customers with a negative account balance (c_acctbal < 0) in silver.customer'
    AS SELECT
        c_custkey, c_name, c_address, c_nationkey, c_phone, c_acctbal, c_mktsegment, c_comment
    FROM {CATALOG}.silver.customer
    WHERE c_acctbal < 0
""")

in_debt_count = spark.table(f"{CATALOG}.silver.customers_in_debt").count()
print(f"View created: {CATALOG}.silver.customers_in_debt ({in_debt_count} customers in debt)")
display(spark.table(f"{CATALOG}.silver.customers_in_debt"))